# Create Gold Layer Tables

In [0]:
from pyspark.sql.functions import avg, sum, max, min, col, current_date, first, last, round

# Read the silver table
silver_df = spark.table("crypto.silver.slv_crypto").filter(col("is_current")==True)

# Market overview Table
gold_market_summary = silver_df.agg(
    sum("market_cap").alias("total_market_cap"),
    sum("total_volume").alias("total_market_volume"),
    avg("price_change_percentage_24h").alias("avg_market_change_24h"),
    max("current_price").alias("highest_price"),
    min("current_price").alias("lowest_price"),
    current_date().alias("snapshot_date")
)
gold_market_summary.display()

assert gold_market_summary.count() > 0, "gold_market_summary is empty"
assert gold_market_summary.filter(col("total_market_cap") <= 0).count() == 0, "Invalid total market cap"

gold_market_summary.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("crypto.gold.tbl_market")

In [0]:
# top 10 cryptocurrencies
gold_top_crypto_df = silver_df.orderBy(
    col("market_cap").desc()
).limit(10)

gold_top_crypto_df.display()

gold_top_crypto_df.write.format("delta").mode("overwrite").saveAsTable("crypto.gold.tbl_top_crypto")


In [0]:

# Liquidity Metrics Table - How actively a coin is traded.
gold_liquidity_df = silver_df.withColumn(
    "volume_marketcap_ratio",
    col("total_volume") / col("market_cap")
)

assert gold_liquidity_df.count() > 0, "Liquidity table is empty"

gold_liquidity_df.display()

gold_liquidity_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("crypto.gold.tbl_liquidity")


In [0]:

# Daily prices 
gold_daily_prices = silver_df.groupBy(
    "coin_id", "symbol", "name",
    col("ingestion_date").alias("price_date")
).agg(
    first("current_price").alias("open_price"),
    last("current_price").alias("close_price"),
    max("current_price").alias("high_price"),
    min("current_price").alias("low_price"),
    avg("total_volume").alias("avg_volume"),
    avg("price_change_percentage_24h").alias("avg_price_change_pct")
).withColumn("avg_volume", round(col("avg_volume"), 2))


assert gold_daily_prices.count() > 0, "gold_daily_prices is empty"

gold_daily_prices.write.format("delta") \
    .mode("overwrite").saveAsTable("crypto.gold.daily_prices")
print("gold_daily_prices OK")

In [0]:
# Market rankings 
gold_market_rankings = silver_df.select(
    "coin_id", "symbol", "name",
    "market_cap_rank", "market_cap", "current_price",
    col("ingestion_date").alias("snapshot_date")
).orderBy("market_cap_rank")


assert gold_market_rankings.count() > 0, "gold_market_rankings is empty"

gold_market_rankings.write.format("delta") \
    .mode("overwrite").saveAsTable("crypto.gold.market_rankings")
print("gold_market_rankings OK")

In [0]:
%sql
--Top Cryptocurrencies by Market Cap
SELECT name as coin,current_price as price, market_cap, total_volume FROM crypto.gold.tbl_top_crypto
ORDER BY market_cap DESC

In [0]:
%sql
SELECT * FROM crypto.gold.market_rankings
LIMIT 10